## 医疗分诊四分类数据集生成

### Notebook 说明

功能：自动构建症状紧急度四分类微调数据集

分类标签：emergency / urgent / routine / self_care

核心亮点：训练集、测试集使用完全不同句式模板，保证模型评估泛化能力，而非记忆句式

数据集用途：DistilBERT LoRA 医疗分诊模型微调

### 1. 导入依赖 & 全局配置

In [3]:
import env_config

In [4]:
from __future__ import annotations

import json
import random
from pathlib import Path

# 固定随机种子，保证数据集可复现
random.seed(42)

# 四分类标签定义
LABELS = ["emergency", "urgent", "routine", "self_care"]

### 2. 医疗症状语料库（四类分级标准样本）

很多公开医疗数据集脱敏严重、无完整描述、有版权问题

自己写 = 干净、无版权、完全可控、分诊逻辑标准

emergency：心梗、中风、大出血、呼吸困难、自杀念头、头部撞击昏迷等 致死致残

urgent：高烧不退、需要缝合、脱水、肾结石、哮喘加重 当天必须就医

routine：偏头痛周期发作、体检、慢病调药 不急

self_care：普通感冒、小擦伤、久坐头痛 完全可以自愈


In [5]:
SYMPTOMS = {
    "emergency": [
        "crushing chest pain radiating to my left arm",
        "sudden difficulty breathing and I'm gasping for air",
        "one side of my face is drooping and my speech is slurred",
        "I'm bleeding heavily and it won't stop after ten minutes",
        "I passed out and just regained consciousness confused",
        "my lips and throat are swelling up and I can barely swallow after eating peanuts",
        "the worst headache of my life that came on all of a sudden",
        "a sudden severe headache along with confusion and blurred vision",
        "a headache that started after hitting my head hard and I feel drowsy",
        "I'm vomiting blood",
        "I have thoughts of ending my life right now",
        "my child swallowed a bottle of pills",
        "severe abdominal pain and I can't stand up straight",
        "I think I'm having a heart attack",
        "sudden weakness on one side of my body",
        "I was in a car accident and my leg looks deformed",
        "a seizure that has lasted more than five minutes",
    ],
    "urgent": [
        "a fever of 103 that has lasted three days",
        "I twisted my ankle and it's swelling fast, maybe broken",
        "persistent vomiting and diarrhea and I feel very dehydrated",
        "a deep cut on my hand that probably needs stitches",
        "an eye injury after something splashed into it",
        "a dog bit me and broke the skin",
        "worsening pain from what I think is a kidney stone",
        "a rash that's spreading fast with a fever",
        "I ran out of my insulin and my blood sugar readings are very high",
        "my asthma inhaler isn't helping and I'm still wheezing",
        "a urinary tract infection with fever and back pain",
        "chest tightness that started after exercise but isn't severe",
        "sudden severe ear pain with hearing loss",
    ],
    "routine": [
        "mild joint stiffness in the mornings for the past few weeks",
        "I'd like to schedule my annual checkup",
        "a small skin rash that isn't spreading or itching much",
        "occasional mild acid reflux after meals",
        "I want to discuss adjusting my blood pressure medication",
        "recurring mild headaches a couple times a month",
        "migraines with light sensitivity that happen every few weeks and respond to my usual medication",
        "a throbbing headache with nausea and light sensitivity that I get occasionally",
        "a bad headache and I am sensitive to light, but I get this every month before my period",
        "a bad headache with sensitivity to light that lines up with my usual migraine pattern",
        "a mole I'd like a doctor to take a look at eventually",
        "ongoing lower back stiffness after sitting all day",
        "I'd like a referral for a routine eye exam",
        "mild seasonal allergies that come back every spring",
        "I want to ask about starting a new exercise routine safely",
        "follow-up on my cholesterol test results from last month",
    ],
    "self_care": [
        "a runny nose and mild sore throat, feels like a common cold",
        "a small paper cut on my finger",
        "mild muscle soreness after a workout yesterday",
        "a slight headache after a long day at the computer",
        "a headache with mild light sensitivity that gets better after resting in a dark room",
        "a bad headache and I am sensitive to light, but resting in a quiet dark room usually helps",
        "a bad headache with sensitivity to light after a stressful day, nothing like before",
        "a minor sunburn on my shoulders",
        "occasional mild heartburn after spicy food",
        "a little bit of dry, itchy skin in winter",
        "mild hiccups that started an hour ago",
        "a stuffy nose from seasonal pollen",
        "slight fatigue after a poor night's sleep",
        "a small bruise from bumping into a table",
        "mild constipation after traveling",
    ],
}

### 3. 句式模板（训练/测试完全隔离，保证泛化性）

训练看陈述句

测试看疑问句

In [6]:
# 训练集模板：陈述句
TEMPLATES_TRAIN = [
    "I have {s}.",
    "I'm dealing with {s}.",
    "I've been experiencing {s}.",
    "For the past hour I've had {s}.",
    "My symptom is {s}.",
    "Lately I've noticed {s}.",
    "I woke up with {s}.",
    "Right now I have {s}.",
]

# 测试集模板：疑问句、担忧句式（和训练集完全不同，模拟真实用户提问）
TEMPLATES_TEST = [
    "Is it serious that I have {s}?",
    "What should I do about {s}?",
    "I'm worried because I have {s}.",
    "Should I see someone about {s}?",
    "Just started having {s}, any advice?",
]

### 4. 数据集生成核心函数

In [7]:
def _generate(templates: list[str]) -> list[dict]:
    """根据模板生成单条样本"""
    rows = []
    for label, phrases in SYMPTOMS.items():
        for phrase in phrases:
            for template in templates:
                rows.append({"text": template.format(s=phrase), "label": label})
    random.shuffle(rows) # 打乱顺序
    return rows


def build_dataset(out_dir: str = "finetuning/data") -> tuple[list[dict], list[dict]]:
    """生成训练集、测试集并保存为jsonl"""
    train_rows = _generate(TEMPLATES_TRAIN)
    test_rows = _generate(TEMPLATES_TEST)

    # 创建输出文件夹
    path = Path(out_dir)
    path.mkdir(parents=True, exist_ok=True)

    # 保存jsonl文件
    (path / "train.jsonl").write_text("\n".join(json.dumps(r) for r in train_rows))
    (path / "test.jsonl").write_text("\n".join(json.dumps(r) for r in test_rows))

    return train_rows, test_rows

5. 执行生成 + 数据集统计可视化

In [8]:
# 生成数据集
train_rows, test_rows = build_dataset()

# 输出统计信息
print(f"✅ 训练集样本总数: {len(train_rows)}")
print(f"✅ 测试集样本总数: {len(test_rows)}")
print("\n📊 各类别样本分布：")
for label in LABELS:
    n_train = sum(1 for r in train_rows if r["label"] == label)
    n_test = sum(1 for r in test_rows if r["label"] == label)
    print(f"  {label:12s} | 训练: {n_train:3d} | 测试: {n_test:2d}")

# 展示前5条训练样本
print("\n🔍 训练集样本示例：")
for idx, item in enumerate(train_rows[:5]):
    print(f"{idx+1}. {item['text']} => {item['label']}")

✅ 训练集样本总数: 488
✅ 测试集样本总数: 305

📊 各类别样本分布：
  emergency    | 训练: 136 | 测试: 85
  urgent       | 训练: 104 | 测试: 65
  routine      | 训练: 128 | 测试: 80
  self_care    | 训练: 120 | 测试: 75

🔍 训练集样本示例：
1. I'm dealing with mild muscle soreness after a workout yesterday. => self_care
2. I'm dealing with a seizure that has lasted more than five minutes. => emergency
3. I'm dealing with my lips and throat are swelling up and I can barely swallow after eating peanuts. => emergency
4. Lately I've noticed my lips and throat are swelling up and I can barely swallow after eating peanuts. => emergency
5. I woke up with chest tightness that started after exercise but isn't severe. => urgent


### 6. 关键项目亮点
1. 数据集解耦设计：训练/测试采用完全不同句式模板，杜绝模型死记模板，评估结果真实反映泛化能力
2. 医疗分级标准：严格按照「急症/紧急/常规/自理」四级医疗分诊逻辑构建样本
3. 无版权风险：自建合成数据集，规避公开医疗数据集版权、脱敏缺陷
4. 均衡样本分布：四类样本均匀生成，避免分类偏置
5. 可复现实验：固定随机种子，数据集每次生成完全一致，方便迭代对比